In [1]:
import os
%pwd
os.chdir("../")
%pwd

'c:\\Users\\HP\\Documents\\LLM_learning\\End-to-end-ML-with-mlflow'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [13]:
from ml_project.constants import *
from ml_project.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(
        self, 
        config_file_path=CONFIG_FILE_PATH, 
        params_file_path=PARAMS_FILE_PATH, 
        schema_file_path=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> modeltrainerconfig:
        config = self.config.model_trainer
        params = self.params.Elasticnet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )

        return model_trainer_config

In [15]:
from ml_project import logger
import pandas as pd
from sklearn.linear_model import ElasticNet
import joblib

In [16]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Loading training data")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        logger.info("Splitting features and target variable")
        X_train = train_data.drop(columns=[self.config.target_column],axis=1)
        y_train = train_data[self.config.target_column]
        X_test = test_data.drop(columns=[self.config.target_column],axis=1)
        y_test = test_data[self.config.target_column]

        logger.info("Training the model")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio,random_state=42)
        model.fit(X_train, y_train)

        logger.info("Saving the trained model")
        joblib.dump(model, self.config.root_dir / self.config.model_name)

In [17]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    raise e

[2026-03-16 16:26:07,817]: INFO:common:yaml file: config\config.yaml loaded successfully]


[2026-03-16 16:26:07,846]: INFO:common:yaml file: params.yaml loaded successfully]
[2026-03-16 16:26:07,855]: INFO:common:yaml file: schema.yaml loaded successfully]
[2026-03-16 16:26:07,858]: INFO:common:created directory at: artifacts]
[2026-03-16 16:26:07,863]: INFO:common:created directory at: artifacts/model_trainer]
[2026-03-16 16:26:07,869]: INFO:2072578443:Loading training data]
[2026-03-16 16:26:07,983]: INFO:2072578443:Splitting features and target variable]
[2026-03-16 16:26:07,995]: INFO:2072578443:Training the model]
[2026-03-16 16:26:08,030]: INFO:2072578443:Saving the trained model]
